In [34]:
import phoenix as px
from phoenix.otel import register
from openinference.instrumentation.langchain import LangChainInstrumentor

# 1. Запускаем сам сервер Phoenix
session = px.launch_app()

# 2. Регистрируем провайдер трассировки (OpenTelemetry)
# Он будет перехватывать данные и отправлять их в локальный Phoenix
tracer_provider = register()

# 3. Включаем "прослушку" именно для LangChain
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)

print(f"Phoenix готов! Дашборд тут: {session.url}")

Existing running Phoenix instance detected! Shutting it down and starting a new instance...
Overriding of current TracerProvider is not allowed
Attempting to instrument while already instrumented


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix
🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: default
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Phoenix готов! Дашборд тут: http://localhost:6006/


In [35]:
import os
from dotenv import load_dotenv

load_dotenv()

# Теперь к ним можно обращаться через стандартный модуль os
api_key = os.getenv("MISTRAL_API_KEY")
print(api_key) 

Lld8KeTo2clg7lurqzk9G0XZlCuGBdhx


In [36]:
import yaml
import re
from typing import List, Set, TypedDict, Annotated, Literal
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END

# Модель для structured_output (как ты и писал)
class Entities(BaseModel):
    items: List[str] = Field(description="Список сгенерированных сущностей")

class SeedPlan(BaseModel):
    length: int = Field(description="Длина seed-строки для значимой цифровой/буквенной основы сущности")
    mode: Literal["digits", "letters", "alnum"] = Field(description="Тип seed-строк: digits, letters или alnum")
    usage_hint: str = Field(description="Короткое правило, где seed может помочь, а где модель должна следовать YAML без навязывания seed")
    reason: str = Field(description="Коротко почему для этой сущности выбран такой mode и length")

# Состояние нашего подграфа
class SubAgentState(TypedDict):
    entity_key: str          # Ключ из YAML (например, 'PASSPORT_RF')
    target_count: int        # Сколько всего нужно уникальных штук
    unique_samples: Set[str] # Наше множество (авто-дедупликация)
    last_batch: List[str]    # Последний выхлоп модели (для логов/проверки)
    iterations: int             # Счетчик итераций

In [37]:
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_mistralai import ChatMistralAI

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.3,   
    check_every_n_seconds=0.1,
    max_bucket_size=1,
)

In [38]:
from langchain_mistralai import ChatMistralAI
from langchain_core.tools import tool
import random
import string

_rng = random.SystemRandom()

@tool
def generate_seed(length: int, mode: str = "digits", count: int = 1) -> list[str]:
    """Генерирует уникальные случайные seed-строки.
    mode: digits | letters | alnum
    """
    if length <= 0 or count <= 0:
        raise ValueError("length and count must be > 0")

    mode = mode.lower()
    if mode == "digits":
        alphabet = string.digits
    elif mode == "letters":
        alphabet = string.ascii_lowercase
    elif mode == "alnum":
        alphabet = string.ascii_lowercase + string.digits
    else:
        raise ValueError("mode must be one of: digits, letters, alnum")

    out = set()
    while len(out) < count:
        out.add("".join(_rng.choice(alphabet) for _ in range(length)))
    return list(out)


# Инициализация модели
llm = ChatMistralAI(model="mistral-small-2506", #mistral-small-2506
                    temperature=0.7,
                    timeout = 60,
                    rate_limiter=rate_limiter,
                    max_concurrent_requests=1,
                    max_retries=3)
seed_planner_llm = llm.with_structured_output(SeedPlan)
structured_llm = llm.with_structured_output(Entities)

#llm = ChatOllama(model="qwen3.5:9b", reasoning = False, format="json", temperature=0.7)
# seed_planner_llm = llm.with_structured_output(SeedPlan)
# structured_llm = llm.with_structured_output(Entities)
# 
# Загрузка промптов (предположим, файл лежит рядом)
with open('../../configs/prompts.yaml', 'r', encoding='utf-8') as f:
    PROMPTS_CONFIG = yaml.safe_load(f)


def generate_batch_node(state: SubAgentState):
    entity_key = state['entity_key']
    config = PROMPTS_CONFIG.get('ENTITIES', {}).get(entity_key)

    if config is None:
        raise ValueError(f"Сущность {entity_key} не найдена в секции ENTITIES в prompts.yaml")

    # Считаем, сколько еще не хватает до цели
    needed = state['target_count'] - len(state['unique_samples'])
    batch_size = min(needed, 10) # Генерим не больше 10 за раз для качества

    # Берем системный промпт и подставляем N
    sys_prompt = config['system_prompt'].format(N=batch_size)

    seed_plan_messages = [
        SystemMessage(content=(
            "Ты подбираешь параметры для генератора seed-строк. "
            "Seed нужен только как источник случайных сырых символов, а не как формат будущих примеров. "
            "Стиль, шум, разделители и обрамление задаются промптом сущности. "
            "Верни только структуру SeedPlan."
        )),
        HumanMessage(content=(
            f"Сущность: {entity_key}\n\n"
            f"Промпт сущности:\n{sys_prompt}\n\n"
            "Выбери mode, length и usage_hint для generate_seed.\n"
            "mode='digits' — если seed полезен в основном для цифр.\n"
            "mode='letters' — если seed полезен в основном для букв.\n"
            "mode='alnum' — если seed полезен для смешанной буквенно-цифровой основы: логины, ID, коды, ссылочные хвосты, email-like части.\n"
            "length — длина seed-строки для значимой цифровой/буквенной основы сущности, исходя из описания в промпте. "
            "Если длина основы переменная, выбери типичную или удобную длину, но не пытайся покрыть весь текст сущности вместе со служебными префиксами/доменами/словами.\n"
            "usage_hint — коротко объясни, где seed может помочь генератору не выдумывать ядро, а где нужно просто следовать YAML. "
            "Seed не обязан быть использован целиком и не должен навязывать формат итоговой сущности."
        ))
    ]
    seed_plan = seed_planner_llm.invoke(seed_plan_messages)

    seed_length = max(6, min(seed_plan.length, 64))
    seed_pool = generate_seed.invoke({
        "length": seed_length,
        "mode": seed_plan.mode,
        "count": batch_size
    })

    # Формируем подсказку, чтобы не повторяться (берем последние 5 примеров)
    history = list(state['unique_samples'])[-5:]
    user_content = f"Сгенерируй {batch_size} новых примеров."
    user_content += (
        f" Если помогает разнообразию, можешь брать отдельные цифры или буквы из этих случайных заготовок: {seed_pool}."
        " Это необязательная подсказка, а не требование: если заготовки мешают реалистичному формату, игнорируй их и генерируй по системному промпту."
        " Главное — корректная сущность и стиль из системного промпта."
        " В батче большинство примеров должны быть валидными и аккуратно записанными: разные допустимые разделители, регистр, префиксы или обрамление — это ок."
        " Действительно шумных вариантов с лишним пробелом, странной пунктуацией, мелкой небрежностью или похожей человеческой ошибкой делай не больше 1-2 на 10 примеров."
    )
    if history:
        user_content += f" Не повторяй эти форматы: {history}. Попробуй другие форматы записи, разделители или цифры."

    messages = [
        SystemMessage(content=sys_prompt),
        HumanMessage(content=user_content)
    ]

    # Вызов модели
    response = structured_llm.invoke(messages)


    return {
        "last_batch": response.items,
        "iterations": state['iterations'] + 1
    }


def validate_and_add_node(state: SubAgentState):
    # Просто добавляем всё, что выдала модель, в set()
    # Так как мы отказались от регулярок, тут только дедупликация
    new_items = state['last_batch']
    updated_samples = state['unique_samples'].copy()

    for item in new_items:
        if item and len(item) > 5: # Базовый фильтр от пустых строк
            updated_samples.add(item)

    return {"unique_samples": updated_samples}


def should_continue(state: SubAgentState):
    # Если набрали нужное количество или превысили лимит попыток (напр. 20)
    if len(state['unique_samples']) >= state['target_count'] or state['iterations'] > 20:
        return "end"
    return "continue"



In [39]:
# Собираем граф
workflow = StateGraph(SubAgentState)

# Добавляем узлы
workflow.add_node("generator", generate_batch_node)
workflow.add_node("validator", validate_and_add_node)

# Устанавливаем точку входа
workflow.set_entry_point("generator")

# Связываем узлы
workflow.add_edge("generator", "validator")

# Добавляем условный переход из валидатора
workflow.add_conditional_edges(
    "validator",
    should_continue,
    {
        "continue": "generator",
        "end": END
    }
)

# Компилируем
sub_agent_app = workflow.compile()

In [40]:
input_state = {
    'entity_key': 'VK',       
    'target_count': 100,
    'unique_samples': set(),
    'iterations': 0            
}
result = sub_agent_app.invoke(input_state,output_keys = ['entity_key', 'unique_samples'])

In [41]:
result

{'entity_key': 'VK',
 'unique_samples': {'@0bcaqs4495',
  '@a1skqdgajn',
  '@ivan_ivanov_msk',
  '@iynbvze88x',
  '@petr_petrov_msk',
  '@petr_petrov_spb',
  '@petrov_petr_msk',
  '@petrov_semen',
  '@raxzwce9tj',
  '[id0lth4vvztw|Друзья]',
  '[id1234567890|Анна]',
  '[id12345678|Контакт]',
  '[id12345678|Мой профиль]',
  '[id192837465|Анна]',
  '[id555666777|Ольга]|vk.me',
  '[id5z6y2xjvrm|Контакт]',
  '[id|Игорь]|vk.com',
  'https://m.vk.com/id9876543210',
  'https://vk.com/ke3x4bz0bn',
  'https://vk.com/sergey_ivanov_spb?ref=profile',
  'https://vk.com/username_123',
  'https://vk.me/id1928374650',
  'https://vk.me/rvqjj44t03',
  'm.vk.com/143669d08q',
  'm.vk.com/4nl9rb7cu2',
  'm.vk.com/4ut6lhg9el',
  'm.vk.com/c0daxevtuy',
  'm.vk.com/emo00zjsnu',
  'm.vk.com/id444555666',
  'm.vk.com/id45678901',
  'm.vk.com/id5566778899',
  'm.vk.com/jiyv1w4twd',
  'vk.com/@maria_petrova',
  'vk.com/[id11121314|Анна]',
  'vk.com/[id111222333|Анна]',
  'vk.com/[id2222222|Анна Петрова]',
  'vk.co

In [39]:
import os
import json

FOLDER_PATH = '../outputs/'

entity_name = result['entity_key']
samples = result['unique_samples']

with open(FOLDER_PATH + f'{entity_name}.json', 'r') as f:
    file = json.load(f)
    print('Current len:', len(file))
new = set(file)
new.update(samples)
print('New len:', len(new))


Current len: 20
New len: 107


In [40]:
with open(FOLDER_PATH + f'{entity_name}.json', 'w') as f:
    json.dump(list(new), f, ensure_ascii = False, indent = 4)

In [31]:
# import os
# import json
# FOLDER_PATH = '../outputs/'
# categories = ['PASSPORT_RF', 'BANK_CARD', 'PHONE_NUMBER', 'EMAIL', 'TELEGRAM', 'VK']
# for cat in categories:
#     input_state = {
#     'entity_key': cat,       
#     'target_count': 10,
#     'unique_samples': set(),
#     'iterations': 0            
#     }
#     res = sub_agent_app.invoke(input_state,output_keys = ['unique_samples'])['unique_samples']
#     print(cat)
#     print(res)
#     with open(FOLDER_PATH + f'{cat}.json', 'w') as f:
#         json.dump(list(res), f, ensure_ascii = False, indent = 4)
#     break
        
    

PASSPORT_RF
{'паспорт сер. 4511 № 123456', '4501 112233', '4612 889900', '4511123456', '4002-987654', '4005 654321', '0118-554433', '7715 001234', '6010-776655', '2014 334455'}
